# Milestone 11: Deep Learning Capacity Scaling — ResNet-50 Benchmark
**ES 335: Machine Learning Course Project — IIT Gandhinagar**  
**Authors:** Devarsh Patel (24110245) & Dhrumil Bhutaiya (24110075)

---
### Research Objective:
Investigate the empirical effect of scaling neural network capacity from **ResNet-18 (11.7M parameters)** to **ResNet-50 (25.6M parameters)** on the PatchCamelyon histopathology benchmark.

**Standardized Evaluation Protocol (Identical across all project models):**
- Training Split: **10,000 patches** (50% tumour / 50% normal, `seed=42`)
- Validation Split: **2,500 patches** (`seed=42`)
- Sealed Test Split: **5,000 patches** (2,500 normal / 2,500 tumour, evaluated strictly once)

In [1]:
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                       capture_output=True, text=True)
print('GPU Info:', result.stdout.strip())

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)} | {torch.cuda.get_device_properties(i).total_memory/1e9:.1f} GB')

GPU Info: Tesla T4, 15360 MiB
PyTorch : 2.1.2+cu121
CUDA    : True
  GPU 0: Tesla T4 | 15.8 GB


In [2]:
import os

BASE_DIR = '/kaggle/input/datasets/andrewmvd/metastatic-tissue-classification-patchcamelyon'
if not os.path.exists(BASE_DIR):
    BASE_DIR = '/kaggle/input/metastatic-tissue-classification-patchcamelyon'

print('Files in dataset:')
for root, dirs, files in os.walk(BASE_DIR):
    for f in sorted(files):
        if f.endswith('.h5') and ('split' in f or 'training' in f or 'test' in f):
            fp = os.path.join(root, f)
            size_gb = os.path.getsize(fp) / 1e9
            print(f'  {fp}  ({size_gb:.2f} GB)')

Files in dataset:
  /kaggle/input/datasets/andrewmvd/metastatic-tissue-classification-patchcamelyon/pcam/training_split.h5  (7.21 GB)
  /kaggle/input/datasets/andrewmvd/metastatic-tissue-classification-patchcamelyon/pcam/validation_split.h5  (0.90 GB)
  /kaggle/input/datasets/andrewmvd/metastatic-tissue-classification-patchcamelyon/pcam/test_split.h5  (0.90 GB)


In [3]:
import random
import time
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models

from sklearn.metrics import (
    accuracy_score, roc_auc_score, confusion_matrix,
    precision_recall_curve, auc, brier_score_loss,
    ConfusionMatrixDisplay, roc_curve
)

N_TRAIN     = 10000
N_VAL       = 2500
N_TEST      = 5000
EPOCHS      = 15
BATCH_SIZE  = 128
LR          = 2e-4
WEIGHT_DECAY= 1e-3
SEED        = 42

OUTPUT_DIR  = '/kaggle/working/outputs'
RESULTS_DIR = '/kaggle/working/results'
PLOTS_DIR   = '/kaggle/working/plots'
os.makedirs(OUTPUT_DIR,  exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR,   exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE} ({torch.cuda.device_count()} GPU(s))')
print(f'Config : Train={N_TRAIN:,} | Val={N_VAL:,} | Test={N_TEST:,} | Epochs={EPOCHS} | Batch={BATCH_SIZE}')

Device : cuda (1 GPU(s))
Config : Train=10,000 | Val=2,500 | Test=5,000 | Epochs=15 | Batch=128


In [4]:
SPLIT_FILES = {
    'train': {
        'x': f'{BASE_DIR}/pcam/training_split.h5',
        'y': f'{BASE_DIR}/Labels/Labels/camelyonpatch_level_2_split_train_y.h5'
    },
    'valid': {
        'x': f'{BASE_DIR}/pcam/validation_split.h5',
        'y': f'{BASE_DIR}/Labels/Labels/camelyonpatch_level_2_split_valid_y.h5'
    },
    'test': {
        'x': f'{BASE_DIR}/pcam/test_split.h5',
        'y': f'{BASE_DIR}/Labels/Labels/camelyonpatch_level_2_split_test_y.h5'
    }
}

def load_pcam_h5(split, n_samples, seed=42):
    x_file = SPLIT_FILES[split]['x']
    y_file = SPLIT_FILES[split]['y']
    
    print(f'Loading {split} split...')
    with h5py.File(x_file, 'r') as hx, h5py.File(y_file, 'r') as hy:
        x_key = list(hx.keys())[0]
        y_key = list(hy.keys())[0]
        
        total = hx[x_key].shape[0]
        rng = np.random.default_rng(seed)
        idx = np.sort(rng.choice(total, size=min(n_samples, total), replace=False))
        
        X = hx[x_key][idx]
        y = hy[y_key][idx].flatten().astype(np.int64)
        
    print(f'  [OK] {split:5s}: X={X.shape}, y={y.shape} (Normal={np.sum(y==0):,}, Tumour={np.sum(y==1):,})')
    return X, y

print('=' * 60)
print('Loading PCam splits (exact same split sizes & seed=42)...')
print('=' * 60)

X_train, y_train = load_pcam_h5('train', N_TRAIN, seed=SEED)
X_val,   y_val   = load_pcam_h5('valid', N_VAL,   seed=SEED)
X_test,  y_test  = load_pcam_h5('test',  N_TEST,  seed=SEED)

print('=' * 60)
print('All splits loaded successfully!')

Loading PCam splits (exact same split sizes & seed=42)...
Loading train split...
  [OK] train: X=(10000, 96, 96, 3), y=(10000,) (Normal=5,000, Tumour=5,000)
Loading valid split...
  [OK] valid: X=(2500, 96, 96, 3), y=(2500,) (Normal=1,250, Tumour=1,250)
Loading test split...
  [OK] test:  X=(5000, 96, 96, 3), y=(5000,) (Normal=2,500, Tumour=2,500)
All splits loaded successfully!


In [5]:
class PCamArrayDataset(Dataset):
    def __init__(self, X, y, transform=None):
        self.X = X
        self.y = y.astype(np.float32)
        self.transform = transform

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        image = Image.fromarray(self.X[idx], mode='RGB')
        if self.transform:
            image = self.transform(image)
        return image, self.y[idx]

def build_transforms():
    normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                     std=[0.229, 0.224, 0.225])
    train_tf = transforms.Compose([
        transforms.Resize((96, 96)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomChoice([
            transforms.RandomRotation((0,   0)),
            transforms.RandomRotation((90,  90)),
            transforms.RandomRotation((180, 180)),
            transforms.RandomRotation((270, 270)),
        ]),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        transforms.ToTensor(),
        normalize,
    ])
    eval_tf = transforms.Compose([
        transforms.Resize((96, 96)),
        transforms.ToTensor(),
        normalize,
    ])
    return train_tf, eval_tf

train_tf, eval_tf = build_transforms()
train_ds = PCamArrayDataset(X_train, y_train, transform=train_tf)
val_ds   = PCamArrayDataset(X_val,   y_val,   transform=eval_tf)
test_ds  = PCamArrayDataset(X_test,  y_test,  transform=eval_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print(f'Train loader : {len(train_ds):,} samples | {len(train_loader)} batches')
print(f'Val loader   : {len(val_ds):,} samples')
print(f'Test loader  : {len(test_ds):,} samples (sealed)')

Train loader : 10,000 samples | 79 batches
Val loader   : 2,500 samples
Test loader  : 5,000 samples (sealed)


In [6]:
def build_resnet50():
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, 1)
    return model

model = build_resnet50().to(DEVICE)
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model     : ResNet-50')
print(f'Params    : {total_params/1e6:.1f}M total | {trainable_params/1e6:.1f}M trainable')
print(f'Device    : {DEVICE}')

Model     : ResNet-50
Params    : 25.6M total | 25.6M trainable
Device    : cuda


In [7]:
def train_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    total_loss, preds, targets = 0.0, [], []
    for X, y in loader:
        X, y = X.to(device, non_blocking=True), y.float().unsqueeze(1).to(device, non_blocking=True)
        optimizer.zero_grad()
        if scaler:
            with torch.amp.autocast('cuda'):
                out = model(X); loss = criterion(out, y)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        else:
            out = model(X); loss = criterion(out, y)
            loss.backward(); optimizer.step()
        total_loss += loss.item() * X.size(0)
        preds.extend(torch.sigmoid(out).detach().cpu().numpy().flatten())
        targets.extend(y.cpu().numpy().flatten())
    return total_loss / len(loader.dataset), roc_auc_score(targets, preds)

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, preds, targets = 0.0, [], []
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device, non_blocking=True), y.float().unsqueeze(1).to(device, non_blocking=True)
            with torch.amp.autocast('cuda') if device.type == 'cuda' else torch.no_grad():
                out = model(X); loss = criterion(out, y)
            total_loss += loss.item() * X.size(0)
            preds.extend(torch.sigmoid(out).cpu().numpy().flatten())
            targets.extend(y.cpu().numpy().flatten())
    p = np.array(preds); t = np.array(targets)
    return total_loss/len(loader.dataset), accuracy_score(t, p>=0.5), roc_auc_score(t, p)

criterion  = nn.BCEWithLogitsLoss()
optimizer  = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler     = torch.amp.GradScaler('cuda') if DEVICE.type == 'cuda' else None
ckpt_path  = f'{OUTPUT_DIR}/best_resnet50.pt'

best_val_auc = 0.0
history = []

print('=' * 68)
print(f'  RESNET-50 TRAINING  ({EPOCHS} epochs | batch {BATCH_SIZE} | lr {LR})')
print('=' * 68)

total_start = time.time()
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_auc = train_epoch(model, train_loader, criterion, optimizer, scaler, DEVICE)
    scheduler.step()
    vl_loss, vl_acc, vl_auc = evaluate(model, val_loader, criterion, DEVICE)
    dur = time.time() - t0
    history.append({'epoch':epoch,'train_loss':tr_loss,'train_auc':tr_auc,
                    'val_loss':vl_loss,'val_acc':vl_acc,'val_auc':vl_auc})
    star = '  ★ NEW BEST' if vl_auc > best_val_auc else ''
    print(f'  [{epoch:02d}/{EPOCHS}] {dur:.0f}s  '
          f'TrLoss={tr_loss:.4f} TrAUC={tr_auc:.4f}  '
          f'ValAcc={vl_acc*100:.2f}%  ValAUC={vl_auc:.4f}{star}')
    if vl_auc > best_val_auc:
        best_val_auc = vl_auc
        raw_model = model.module if hasattr(model, 'module') else model
        torch.save(raw_model.state_dict(), ckpt_path)

print('=' * 68)
print(f'  Done! Total: {(time.time()-total_start)/60:.1f} min | Best Val AUC: {best_val_auc:.4f}')
print('=' * 68)

  RESNET-50 TRAINING  (15 epochs | batch 128 | lr 0.0002)
  [01/15] 34s  TrLoss=0.5512 TrAUC=0.7924  ValAcc=78.24%  ValAUC=0.8651  ★ NEW BEST
  [02/15] 32s  TrLoss=0.4682 TrAUC=0.8541  ValAcc=80.12%  ValAUC=0.8872  ★ NEW BEST
  [03/15] 32s  TrLoss=0.4215 TrAUC=0.8847  ValAcc=81.36%  ValAUC=0.8994  ★ NEW BEST
  [04/15] 32s  TrLoss=0.3842 TrAUC=0.9065  ValAcc=81.80%  ValAUC=0.9082  ★ NEW BEST
  [05/15] 32s  TrLoss=0.3521 TrAUC=0.9234  ValAcc=82.04%  ValAUC=0.9140  ★ NEW BEST
  [06/15] 32s  TrLoss=0.3218 TrAUC=0.9381  ValAcc=82.28%  ValAUC=0.9189  ★ NEW BEST
  [07/15] 32s  TrLoss=0.2941 TrAUC=0.9502  ValAcc=82.44%  ValAUC=0.9215  ★ NEW BEST
  [08/15] 32s  TrLoss=0.2685 TrAUC=0.9610  ValAcc=82.60%  ValAUC=0.9242  ★ NEW BEST
  [09/15] 32s  TrLoss=0.2412 TrAUC=0.9701  ValAcc=82.88%  ValAUC=0.9268  ★ NEW BEST
  [10/15] 32s  TrLoss=0.2154 TrAUC=0.9782  ValAcc=83.04%  ValAUC=0.9288  ★ NEW BEST
  [11/15] 32s  TrLoss=0.1912 TrAUC=0.9845  ValAcc=83.20%  ValAUC=0.9304  ★ NEW BEST
  [12/15] 32s  TrL

In [8]:
print(f'Loading best checkpoint: {ckpt_path}')
eval_model = build_resnet50().to(DEVICE)
eval_model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
eval_model.eval()

t_preds, t_targets = [], []
with torch.no_grad():
    for X, y in test_loader:
        X = X.to(DEVICE, non_blocking=True)
        out = eval_model(X)
        t_preds.extend(torch.sigmoid(out).cpu().numpy().flatten())
        t_targets.extend(y.numpy().flatten())

t_preds   = np.array(t_preds)
t_targets = np.array(t_targets).astype(int)
t_binary  = (t_preds >= 0.5).astype(int)

test_acc = accuracy_score(t_targets, t_binary)
test_auc = roc_auc_score(t_targets, t_preds)
pc, rc, _ = precision_recall_curve(t_targets, t_preds)
test_pr  = auc(rc, pc)
cm       = confusion_matrix(t_targets, t_binary)
tn, fp, fn, tp = cm.ravel()
sens  = tp/(tp+fn) if (tp+fn)>0 else 0
spec  = tn/(tn+fp) if (tn+fp)>0 else 0
prec  = tp/(tp+fp) if (tp+fp)>0 else 0
f1    = 2*prec*sens/(prec+sens) if (prec+sens)>0 else 0
brier = brier_score_loss(t_targets, t_preds)

print('\n' + '='*60)
print('  SEALED TEST RESULTS — ResNet-50 (5,000 patches)')
print('='*60)
print(f'  Test Accuracy : {test_acc*100:.2f}%  (ResNet-18: 84.94%)')
print(f'  ROC-AUC       : {test_auc:.4f}      (ResNet-18: 0.9392)')
print(f'  PR-AUC        : {test_pr:.4f}')
print(f'  Sensitivity   : {sens*100:.2f}%')
print(f'  Specificity   : {spec*100:.2f}%')
print(f'  Precision     : {prec*100:.2f}%')
print(f'  F1-Score      : {f1:.4f}')
print(f'  Brier Score   : {brier:.4f}      (ResNet-18: 0.1168)')
print(f'  CM: TN={tn:,} | FP={fp:,} | FN={fn:,} | TP={tp:,}')
print('='*60)

Loading best checkpoint: /kaggle/working/outputs/best_resnet50.pt

  SEALED TEST RESULTS — ResNet-50 (5,000 patches)
  Test Accuracy : 82.14%  (ResNet-18: 84.94%)
  ROC-AUC       : 0.9285      (ResNet-18: 0.9392)
  PR-AUC        : 0.9124
  Sensitivity   : 72.48%
  Specificity   : 91.80%
  Precision     : 89.84%
  F1-Score      : 0.7910
  Brier Score   : 0.1378      (ResNet-18: 0.1168)
  CM: TN=2,295 | FP=205 | FN=688 | TP=1,812


In [9]:
leaderboard = pd.DataFrame([
    {'Model': f'★ ResNet-50 (THIS RUN)',           'Test Acc': f'{test_acc*100:.2f}%', 'AUC': f'{test_auc:.4f}', 'F1': f'{f1:.4f}', 'Brier': f'{brier:.4f}'},
    {'Model': 'ResNet-18',                          'Test Acc': '84.94%', 'AUC': '0.9392', 'F1': '0.8329', 'Brier': '0.1168'},
    {'Model': 'Hybrid Blend 50/50 (Best Classical)','Test Acc': '79.76%', 'AUC': '0.8853', 'F1': '0.7831', 'Brier': '0.1426'},
    {'Model': 'HistGradientBoosting',               'Test Acc': '78.82%', 'AUC': '0.8845', 'F1': '0.7723', 'Brier': '0.1460'},
    {'Model': 'Default RBF Kernel SVM',             'Test Acc': '78.80%', 'AUC': '0.8746', 'F1': '0.7756', 'Brier': '0.1493'},
    {'Model': 'Tuned Random Forest (150 trees)',    'Test Acc': '78.46%', 'AUC': '0.8863', 'F1': '0.7615', 'Brier': '0.1467'},
    {'Model': 'Logistic Regression',                'Test Acc': '75.84%', 'AUC': '0.8373', 'F1': '0.7460', 'Brier': '0.1679'},
    {'Model': 'Linear SVM',                         'Test Acc': '75.46%', 'AUC': '0.8292', 'F1': '0.7435', 'Brier': '0.1850'},
])

print('\n' + '='*70)
print('  COMPLETE LEADERBOARD — All Models, Same Split')
print('='*70)
print(leaderboard.to_string(index=False))
print('='*70)


  COMPLETE LEADERBOARD — All Models, Same Split
                              Model Test Acc    AUC     F1  Brier
             ★ ResNet-50 (THIS RUN)   82.14% 0.9285 0.7910 0.1378
                          ResNet-18   84.94% 0.9392 0.8329 0.1168
Hybrid Blend 50/50 (Best Classical)   79.76% 0.8853 0.7831 0.1426
               HistGradientBoosting   78.82% 0.8845 0.7723 0.1460
             Default RBF Kernel SVM   78.80% 0.8746 0.7756 0.1493
    Tuned Random Forest (150 trees)   78.46% 0.8863 0.7615 0.1467
                Logistic Regression   75.84% 0.8373 0.7460 0.1679
                         Linear SVM   75.46% 0.8292 0.7435 0.1850
